# 04 · Baseline SOH regression

**Goal:** establish simple, reproducible SOH baselines on `ml_training_dataset.csv` before any continual learning.

Design choices:
* **Time-aware split per battery.** Each battery's cycles are sorted by `Cycle_Date` (then `Cycle_Index`); the first 70 % go to train, the next 15 % to validation, the last 15 % to test. Nothing is shuffled, so the model is always evaluated on a battery's *later* life.
* **Battery groups evaluated separately.** Metrics are reported per `Battery_ID` plus an overall row.
* **No target leakage.** `RPT`, `RPT_Number`, `Capacity_Ah`, `Reference_Capacity_Ah`, `Days_Since_RPT` are excluded from the inputs (see notebook 03).
* **Missing temperatures** are left as NaN in the data; the linear model uses a train-fitted median plus missing-indicator columns, the tree model handles NaN natively.

Metrics: RMSE, MAE (percentage points of SOH), MAPE (%) and R².

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CONFIG = {
    "dataset": "ml_training_dataset.csv",
    "target": "SOH_Percent",
    "split": {"method": "chronological per battery", "train_frac": 0.70, "val_frac": 0.15, "order_by": ["Cycle_Date", "Cycle_Index"]},
    "ridge_alphas": [0.01, 0.1, 1.0, 10.0, 100.0],
    "hgb_params": {"max_iter": 300, "learning_rate": 0.05, "max_leaf_nodes": 15, "min_samples_leaf": 20, "random_state": 0},
    "random_state": 0,
}
np.random.seed(CONFIG["random_state"])

## 2. Load the supervised dataset

Only `ml_training_dataset.csv` is required: it contains cycles with a known SOH from the most recent RPT on or before each cycle.

In [ ]:
dp.check_data_files(DATA_ROOT, required=[CONFIG["dataset"]])
data = dp.read_dataset(DATA_ROOT / CONFIG["dataset"])
data[dp.GROUP_COL] = data[dp.GROUP_COL].astype(str)
TARGET = CONFIG["target"]

n_missing_target = int(data[TARGET].isna().sum())
if n_missing_target:
    raise ValueError(f"{n_missing_target} rows in {CONFIG['dataset']} have no {TARGET}; expected 0 (see notebook 03).")
print("shape:", data.shape)
print("battery groups:", dp.check_battery_groups(data[dp.GROUP_COL]))

## 3. Time-aware split per battery

After splitting we verify, per battery, that every training cycle is dated no later than every validation cycle, and validation no later than test.

Because SOH is a step function that only changes at RPTs, the first test cycles often share an RPT (and therefore an identical label) with the last training cycles. The table reports how many test rows fall in an RPT interval already seen in training — a measure of how much of the test set is "interpolation within a known RPT" versus genuinely new health states.

In [ ]:
split_df = dp.temporal_split(
    data,
    train_frac=CONFIG["split"]["train_frac"],
    val_frac=CONFIG["split"]["val_frac"],
    time_cols=CONFIG["split"]["order_by"],
)
ordering = dp.check_split_ordering(split_df)
if not (ordering["train_before_val"].all() and ordering["val_before_test"].all()):
    raise AssertionError("Temporal split violated ordering; check for duplicate or unsorted dates.")

train_rpts = split_df[split_df["split"] == "train"].groupby(dp.GROUP_COL)["RPT"].apply(set)
split_df["rpt_seen_in_train"] = [r in train_rpts.get(b, set()) for b, r in zip(split_df[dp.GROUP_COL], split_df["RPT"])]
overlap = split_df[split_df["split"] == "test"].groupby(dp.GROUP_COL)["rpt_seen_in_train"].mean().mul(100).round(1)

ordering["test_rows_in_train_rpt_%"] = overlap
ordering["test_distinct_rpts"] = split_df[split_df["split"] == "test"].groupby(dp.GROUP_COL)["RPT"].nunique()
ordering.reindex([g for g in dp.BATTERY_GROUPS if g in ordering.index])

## 4. Features

Numeric cycling features only; identifiers, time stamps and every RPT/health column are excluded.

In [ ]:
FEATURES = dp.model_feature_columns(split_df, exclude=["rpt_seen_in_train"])
print(f"{len(FEATURES)} input features:", FEATURES)
leaks = set(FEATURES) & set(dp.HEALTH_COLUMNS)
assert not leaks, f"leakage columns in inputs: {leaks}"

train = split_df[split_df["split"] == "train"]
val = split_df[split_df["split"] == "val"]
test = split_df[split_df["split"] == "test"]
X_train, y_train = train[FEATURES], train[TARGET]
print({s: len(d) for s, d in [("train", train), ("val", val), ("test", test)]})
print("missing values in training inputs:", {k: int(v) for k, v in X_train.isna().sum().items() if v})

## 5. Baseline models

1. **Persistence** (reference only) — predict the last SOH observed in the battery's training period. It uses battery identity and no features; any learned model should beat it on later life. It is *not* a selection candidate: if the validation window falls inside the last training RPT interval, its validation error is trivially zero.
2. **Ridge regression** — median imputation + missing indicators → standardisation → Ridge. `alpha` is selected on validation RMSE. All transforms are fitted on the training split only.
3. **Histogram gradient boosting** — non-linear baseline with native missing-value handling and fixed hyper-parameters (no tuning).

Note: tree models cannot extrapolate below the lowest SOH seen in training, which matters for a degrading target evaluated on later life.

In [ ]:
predictions = {}

# 1. Persistence
last_train_soh = train.sort_values([dp.GROUP_COL, "Cycle_Date", "Cycle_Index"]).groupby(dp.GROUP_COL)[TARGET].last()
predictions["persistence"] = split_df[dp.GROUP_COL].map(last_train_soh).to_numpy()

# 2. Ridge with alpha chosen on validation
ridge_search = []
for alpha in CONFIG["ridge_alphas"]:
    model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    ridge_search.append({"alpha": alpha, "val_RMSE": dp.regression_metrics(val[TARGET], model.predict(val[FEATURES]))["RMSE"]})
ridge_search = pd.DataFrame(ridge_search)
best_alpha = float(ridge_search.loc[ridge_search["val_RMSE"].idxmin(), "alpha"])
display(ridge_search)
ridge = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), Ridge(alpha=best_alpha)).fit(X_train, y_train)
predictions["ridge"] = ridge.predict(split_df[FEATURES])
print("selected ridge alpha:", best_alpha)

# 3. Gradient boosting
hgb = HistGradientBoostingRegressor(**CONFIG["hgb_params"]).fit(X_train, y_train)
predictions["hist_gradient_boosting"] = hgb.predict(split_df[FEATURES])

## 6. Evaluation per battery group

Metrics for each model on the validation and test periods, per battery and overall (`ALL`). RMSE and MAE (SOH percentage points) are the primary metrics. R² is `NaN` when a battery's target is constant within the evaluated period (a single RPT interval) and is strongly negative when the period spans only one or two RPT values, even for small errors; it is therefore interpreted only for pooled (`ALL`) results (decision D8).

In [ ]:
metric_frames = []
for model_name, pred in predictions.items():
    scored = split_df.assign(pred=pred)
    for split in ("train", "val", "test"):
        m = dp.metrics_by_group(scored[scored["split"] == split], TARGET, "pred")
        metric_frames.append(m.assign(model=model_name, split=split).rename_axis(dp.GROUP_COL).reset_index())
metrics = pd.concat(metric_frames, ignore_index=True)
metrics["n"] = metrics["n"].astype(int)

overall = metrics[metrics[dp.GROUP_COL] == "ALL"].pivot(index="model", columns="split", values=["RMSE", "MAE", "MAPE", "R2"])
display(overall.round(3))

test_by_group = metrics[(metrics["split"] == "test")].pivot(index=dp.GROUP_COL, columns="model", values=["RMSE", "MAE", "MAPE", "R2"])
test_by_group = test_by_group.reindex([g for g in [*dp.BATTERY_GROUPS, "ALL"] if g in test_by_group.index])
test_by_group.round(3)

In [ ]:
# Persistence is a reference, not a candidate: when the validation window lies inside the last
# training RPT interval its validation error is trivially ~0, which says nothing about later life.
val_rmse = metrics[(metrics[dp.GROUP_COL] == "ALL") & (metrics["split"] == "val") & (metrics["model"] != "persistence")].set_index("model")["RMSE"]
BEST_MODEL = val_rmse.idxmin()
print("learned model selected on validation RMSE:", BEST_MODEL, val_rmse.round(3).to_dict())

groups = [g for g in dp.BATTERY_GROUPS if g in split_df[dp.GROUP_COL].unique()]
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=False)
for ax, g in zip(axes.ravel(), groups):
    sub = split_df[split_df[dp.GROUP_COL] == g]
    idx = sub.index
    ax.plot(sub["Cycle_Date"], sub[TARGET], color="#3d3d3a", linewidth=1.5, label="measured SOH (RPT)")
    ax.plot(sub["Cycle_Date"], predictions[BEST_MODEL][idx], color=GROUP_COLORS[g], linewidth=1.2, alpha=0.9, label=BEST_MODEL)
    ax.plot(sub["Cycle_Date"], predictions["persistence"][idx], color="#8a8981", linewidth=1, linestyle="--", label="persistence")
    for s in ("val", "test"):
        start = sub.loc[sub["split"] == s, "Cycle_Date"].min()
        if pd.notna(start):
            ax.axvline(start, color="#b5b4ac", linewidth=1)
            ax.text(start, 1.01, f" {s}", transform=ax.get_xaxis_transform(), fontsize=8, color="#5f5e5a")
    ax.set_title(g, loc="left")
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("SOH (%)")
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Measured vs predicted SOH over each battery's life (train | val | test)")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 7. Save small metrics and configuration

Only a small JSON file (configuration, feature list, per-group metrics, a data fingerprint, library versions and the Git commit) is written to `results/`. Numbers intended for the paper must come from a run on a clean commit (`git_dirty: false`). No models, predictions or data extracts are saved.

In [ ]:
import json
from datetime import datetime, timezone

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "environment": dp.environment_info(REPO_ROOT),
    "config": CONFIG | {"ridge_selected_alpha": best_alpha},
    "features": FEATURES,
    "selected_learned_model_on_val_rmse": BEST_MODEL,
    "data_fingerprint": {
        "rows": int(len(data)),
        "rows_per_group": data[dp.GROUP_COL].value_counts().sort_index().to_dict(),
        "cycle_date_min": str(data["Cycle_Date"].min()),
        "cycle_date_max": str(data["Cycle_Date"].max()),
        "split_rows": split_df["split"].value_counts().to_dict(),
    },
    "metrics": metrics.replace({np.nan: None}).to_dict(orient="records"),
}
RESULTS_DIR.mkdir(exist_ok=True)
out_path = RESULTS_DIR / "baseline_metrics.json"
out_path.write_text(json.dumps(report, indent=2, default=str))
print("wrote", out_path.relative_to(REPO_ROOT), f"({out_path.stat().st_size} bytes)")
print("environment:", report["environment"])
if report["environment"]["git_dirty"] is not False:
    print("WARNING: code is not a clean Git commit; these metrics cannot be traced to a single commit. "
          "Commit changes and re-run before reporting results.")

## 8. Summary

In [ ]:
best_test = metrics[(metrics["model"] == BEST_MODEL) & (metrics["split"] == "test")].set_index(dp.GROUP_COL)
pers_test = metrics[(metrics["model"] == "persistence") & (metrics["split"] == "test")].set_index(dp.GROUP_COL)
pd.DataFrame(
    [
        ("Rows / features", f"{len(data):,} / {len(FEATURES)}"),
        ("Split", f"train {len(train):,} · val {len(val):,} · test {len(test):,} (chronological per battery)"),
        ("Split ordering verified", bool(ordering["train_before_val"].all() and ordering["val_before_test"].all())),
        ("Selected learned model (val RMSE)", BEST_MODEL),
        ("Test RMSE / MAE / MAPE / R² (ALL)",
         " / ".join(f"{best_test.loc['ALL', k]:.3f}" for k in ("RMSE", "MAE", "MAPE", "R2"))),
        ("Persistence test RMSE (ALL)", f"{pers_test.loc['ALL', 'RMSE']:.3f}"),
        ("Worst test group (RMSE)", best_test.drop(index="ALL")["RMSE"].idxmax()),
        ("Mean share of test rows in an RPT seen in training (%)", float(ordering["test_rows_in_train_rpt_%"].mean().round(1))),
    ],
    columns=["item", "value"],
)

### Caveats and next steps

* **Step-function target.** Cycle-level SOH repeats the most recent RPT value, so many consecutive cycles share one label and test rows can share an RPT interval with training rows (see section 3). Scores on those rows partly measure interpolation within a known health state. An RPT-boundary split, or evaluation on RPT-level targets, is a stricter follow-up.
* **Within-battery split.** Every battery appears in training here. This baseline therefore does not measure generalisation to an *unseen* battery group; that is the continual-learning setting (sequential groups, adaptation + retention metrics) to be addressed next.
* **Extrapolation.** Test periods are later in life (lower SOH). The tree model cannot predict below the training range; the linear model can but may drift.
* **Temperature gaps** are imputed only inside the linear pipeline (median fitted on train + indicator); the stored data is unchanged.
* **Literature targets** (RMSE ≈ 0.6–1.5 %) come from different datasets and SOH definitions (here SOH is relative to the first observed RPT, not nominal capacity) and are not directly comparable.